# HUẤN LUYỆN M2 (GraphCodeBERT) MAX_LEN 512 TRÊN BIGCLONEBENCH JAVA (GPU)

Notebook này thực hiện huấn luyện mô hình **M2 (GraphCodeBERT)** trên BigCloneBench Java với độ dài token tối đa **max_len = 512**.

### Thông tin cấu hình:
- **Mã mô hình:** M2
- **Tên phương pháp:** GraphCodeBERT
- **File cấu hình:** `configs/exp003_graphcodebert_bcb.yaml`
- **Tokenizer max_length:** 512 (dynamic padding)
- **Dataset liên kết:** `semanticcodeclonedetection-dataset-m4`
- **Tự động lưu checkpoint & kết quả riêng biệt:** Đảm bảo khi một mô hình kết thúc sẽ ngay lập tức đồng bộ kết quả vào `/kaggle/working/`

---

In [ ]:
# 1. THIẾT LẬP MÔI TRƯỜNG & HÀM BỔ TRỢ STREAMING LOG
import base64, io, json, os, shutil, subprocess, sys, tarfile, time, yaml
from pathlib import Path

def sh(cmd, **kw):
    print(f"\n$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    log_file = Path("/kaggle/working/live_progress.log")
    with log_file.open("a", encoding="utf-8") as lf:
        for line in iter(proc.stdout.readline, ''):
            print(line, end='', flush=True)
            lf.write(line)
            lf.flush()
    proc.stdout.close()
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"Lệnh thất bại (rc={rc}): {cmd}")
    return ""

print("Python:", sys.version.split()[0], flush=True)

In [ ]:
# 2. GIẢI NÉN MÃ NGUỒN DỰ ÁN MỚI NHẤT (SELF-CONTAINED)
WORK_DIR = Path("/kaggle/working/project")
WORK_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_FILE = "configs/exp003_graphcodebert_bcb.yaml"
BUNDLE_B64 = "..."

print(f"Giải nén mã nguồn ({len(BUNDLE_B64)/1024:.1f} KB) vào {WORK_DIR}...")
buf = io.BytesIO(base64.b64decode(BUNDLE_B64))
with tarfile.open(fileobj=buf, mode="r:gz") as tar:
    tar.extractall(WORK_DIR)

os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
sh(f"ls -la {CONFIG_FILE}")

In [ ]:
# 3. CÀI ĐẶT THƯ VIỆN & XÁC THỰC GPU
sh("pip install -q 'tree-sitter>=0.23.0' 'tree-sitter-java>=0.23.5' 'tree-sitter-cpp>=0.23.0' 'datasets>=2.14.0' 'transformers>=4.30.0' 'pyarrow' 'tqdm' 'pyyaml' 'scikit-learn'")

import torch
assert torch.cuda.is_available(), "Cần bật GPU trong Settings -> Accelerator -> GPU (T4/P100)!"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
print(f"GPU SẴN SÀNG: {gpu_name} | VRAM: {vram_gb} GB")
sh("nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader")

In [ ]:
# 4. LIÊN KẾT ĐẶC TRƯNG TỪ DATASET NẾU CÓ
in_dir = Path("/kaggle/input/semanticcodeclonedetection-dataset-m4")
if in_dir.exists():
    print(f"Dataset đính kèm: {in_dir}")
    sh(f"ls -lh {in_dir}")
else:
    print("Chưa phát hiện dataset đính kèm, sử dụng dữ liệu trực tiếp.")

In [ ]:
# 5. TẢI CÁC SPLIT DỮ LIỆU ĐỂ HUẤN LUYỆN (TRAIN 50K, VAL 5K, TEST 5K)
data_raw = Path("data/raw/bigclonebench")
data_raw.mkdir(parents=True, exist_ok=True)
print("Tải các phân tập dữ liệu BigCloneBench (Train: 50k, Val: 5k, Test: 5k)...")
sh("python scripts/download_bcb_subsets.py --split train --max-samples 50000")
sh("python scripts/download_bcb_subsets.py --split validation --max-samples 5000")
sh("python scripts/download_bcb_subsets.py --split test --max-samples 5000")
sh("ls -lh data/raw/bigclonebench/")

In [ ]:
# 6. PREFLIGHT & DIAGNOSTIC CHECK
print("--- PREFLIGHT M2 ---")
sh(f"python -m src.experiments.bcb_runner --config {CONFIG_FILE} --mode preflight --device cuda")
print("\n--- DIAGNOSTIC CHECK M2 ---")
sh(f"python -m src.experiments.bcb_runner --config {CONFIG_FILE} --mode diagnostic --device cuda")

In [ ]:
# 7. TIẾN HÀNH HUẤN LUYỆN M2 (GraphCodeBERT) TRÊN GPU
print("=" * 60)
print("BẮT ĐẦU HUẤN LUYỆN CHÍNH THỨC M2 (GraphCodeBERT) MAX_LEN 512...")
print("=" * 60)
TIME_BUDGET_S = 9 * 3600
try:
    sh(f"python -m src.experiments.bcb_runner --config {CONFIG_FILE} --mode train --device cuda --time-budget-s {TIME_BUDGET_S}")
finally:
    # Lưu ngay kết quả huấn luyện (dù evaluate có lỗi thì kết quả train vẫn còn nguyên)
    with open(CONFIG_FILE, 'r', encoding='utf-8') as f:
        exp_id = yaml.safe_load(f)['experiment']['id'].lower().replace('-', '_')
    res_dir = Path('results') / exp_id
    dest_dir = Path('/kaggle/working/m2_results')
    dest_dir.mkdir(parents=True, exist_ok=True)
    if res_dir.exists():
        for f in res_dir.glob('*'):
            sh(f'cp -r {f} {dest_dir}/')
            if f.suffix in ('.json', '.pt'):
                sh(f'cp {f} /kaggle/working/')
    sh(f'tar -czf /kaggle/working/m2_results.tar.gz results/')
    print(f'>>> ĐÃ SAO LƯU CHECKPOINT VÀ KẾT QUẢ HUẤN LUYỆN CỦA M2 VÀO /kaggle/working/')

In [ ]:
# 8. ĐÁNH GIÁ MÔ HÌNH M2 TRÊN TẬP TEST
print("=" * 60)
print("ĐÁNH GIÁ MÔ HÌNH M2 TRÊN TẬP TEST...")
print("=" * 60)
sh(f"python -m src.experiments.bcb_runner --config {CONFIG_FILE} --mode evaluate --device cuda")

In [ ]:
# 9. ĐÓNG GÓI & TỔNG KẾT KẾT QUẢ VÀO /kaggle/working/
with open(CONFIG_FILE, 'r', encoding='utf-8') as f:
    exp_id = yaml.safe_load(f)['experiment']['id'].lower().replace('-', '_')
res_dir = Path('results') / exp_id
dest_dir = Path('/kaggle/working/m2_results')
dest_dir.mkdir(parents=True, exist_ok=True)

if res_dir.exists():
    for f in res_dir.glob('*'):
        sh(f'cp -r {f} {dest_dir}/')
        if f.suffix in ('.json', '.pt'):
            sh(f'cp {f} /kaggle/working/')

TAR_RESULTS = Path('/kaggle/working/m2_results.tar.gz')
sh(f'tar -czf {TAR_RESULTS} results/')
sh('ls -lh /kaggle/working/*.json /kaggle/working/*.tar.gz 2>/dev/null || ls -lh /kaggle/working/')

if (res_dir / 'test_metrics.json').exists():
    metrics = json.loads((res_dir / 'test_metrics.json').read_text(encoding='utf-8'))
    print('\nBẢNG TEST METRICS CỦA M2 (max_len=512):\n', json.dumps(metrics, indent=2))
print('\nHUẤN LUYỆN VÀ ĐÁNH GIÁ M2 HOÀN TẤT THÀNH CÔNG!')